In [33]:
"""
Monte Carlo methods for estimating the second eigenvalue and eigenvector
of the Burnside process Markov chain.

Two approaches:
1. Power method on (P - uniform): converges to second eigenvector
2. Autocorrelation method: second eigenvalue from mixing time

The second eigenvalue β₂ controls mixing: ||P^t - π|| ~ β₂^t
"""

import numpy as np
from collections import defaultdict
import itertools
from sage.all import *

# =============================================================================
# Parameters - set these
# =============================================================================
q_val = 5  # Field size
n_val = 4  # Dimension (S_n)

# =============================================================================
# Copy the essential parts from your sampler
# =============================================================================

q = q_val
F = GF(q)

# --- Green polynomials (from your code) ---
from collections import Counter

class FactorialCalculator:
    def __init__(self):
        self.cache = {0: Integer(1), 1: Integer(1)}
    
    def get(self, n):
        if n in self.cache:
            return self.cache[n]
        current = n
        while current not in self.cache:
            current -= 1
        value = self.cache[current]
        for i in range(current + 1, n + 1):
            value *= i
            self.cache[i] = value
        return value

class BinomialCoefficientCalculator:
    def __init__(self):
        self.cache = {}
    
    def get(self, n, k):
        if k < 0 or k > n:
            return Integer(0)
        if k == 0 or k == n:
            return Integer(1)
        k = min(k, n - k)
        key = (n,k)
        if key in self.cache:
            return self.cache[key]
        result = self.get(n-1, k-1) + self.get(n-1, k)
        self.cache[key] = result
        return result

class Partition:
    def __init__(self, parts):
        parts = tuple(sorted([Integer(p) for p in parts if p > 0], reverse=True))
        self.parts = parts

    def __len__(self):
        return len(self.parts)

    @property
    def sum(self):
        return sum(self.parts)

    def element_at(self, i):
        return self.parts[i]

    def to_row_count_dict(self):
        return dict(Counter(self.parts))

    def delete_first_row(self):
        return Partition(self.parts[1:]) if len(self.parts) > 1 else Partition(())

    def concat(self, other):
        return Partition(self.parts + tuple(other.parts))

    def to_key(self):
        return self.parts

    def subpartitions(self):
        counts = self.to_row_count_dict()
        sizes = list(counts.items())
        def rec(i, chosen):
            if i == len(sizes):
                parts = []
                for s, m in chosen:
                    parts.extend([s]*m)
                yield Partition(parts)
            else:
                s, m = sizes[i]
                for take in range(m+1):
                    chosen.append((s,take))
                    yield from rec(i+1, chosen)
                    chosen.pop()
        yield from rec(0, [])

class Polynomial:
    def __init__(self, coeffs=None):
        self.R = PolynomialRing(QQ, 'x')
        self.x = self.R.gen()
        if coeffs is None:
            self.poly = self.R(0)
        elif isinstance(coeffs, dict):
            self.poly = sum(QQ(c)*self.x**d for d,c in coeffs.items())
        else:
            self.poly = sum(QQ(c)*self.x**i for i,c in enumerate(coeffs))

    @staticmethod
    def zero():
        return Polynomial({})

    @staticmethod
    def one():
        return Polynomial({0:1})

    def degree(self):
        return self.poly.degree()

    def __add__(self, other):
        p = Polynomial()
        p.poly = self.poly + other.poly
        return p

    def __sub__(self, other):
        p = Polynomial()
        p.poly = self.poly - other.poly
        return p

    def __mul__(self, other):
        p = Polynomial()
        p.poly = self.poly * other.poly
        return p

    def scalar_multiply(self, r):
        p = Polynomial()
        p.poly = self.poly * QQ(r)
        return p

    def involution(self):
        d = self.degree()
        if d < 0:
            return Polynomial.zero()
        coeffs = {i:self.poly[d-i] for i in range(d+1)}
        return Polynomial(coeffs)

    def __repr__(self):
        return str(self.poly)

def generate_partitions(n):
    if n==0:
        yield Partition([])
        return
    for p in Partitions(n):
        yield Partition(list(p))

class Calculator:
    def __init__(self):
        self._fact = FactorialCalculator()
        self._binom = BinomialCoefficientCalculator()
        self._basic_polys = {}
        self._partition_polys = {}
        self._centralizers = {}
        self._green_polys = {}

    def factorial(self,n): return self._fact.get(n)
    def binomial(self,n,k): return self._binom.get(n,k)

    def get_basic_polynomial(self,j):
        if j in self._basic_polys:
            return self._basic_polys[j]
        coeffs = [1] + [0]*(j-1) + [-1]
        p = Polynomial(coeffs)
        self._basic_polys[j] = p
        return p

    def centralizer_size(self, part):
        key = part.to_key()
        if key in self._centralizers:
            return self._centralizers[key]
        res = Integer(1)
        for s,m in part.to_row_count_dict().items():
            res *= s**m * self.factorial(m)
        self._centralizers[key] = res
        return res

    def z_inverse(self, part):
        key = part.to_key()
        if key in self._partition_polys:
            return self._partition_polys[key]
        res = Polynomial.one()
        for p in part.parts:
            res = res * self.get_basic_polynomial(p)
        res = res.scalar_multiply(1/self.centralizer_size(part))
        self._partition_polys[key] = res
        return res

    def green_polynomial_X(self, lam, mu):
        key = (lam.to_key(), mu.to_key())
        if key in self._green_polys:
            return self._green_polys[key]
        if lam.sum != mu.sum:
            self._green_polys[key] = Polynomial.zero()
            return Polynomial.zero()
        n = lam.sum
        if len(lam)<=1:
            self._green_polys[key] = Polynomial.one()
            return Polynomial.one()
        result = Polynomial.zero()
        mu_count = mu.to_row_count_dict()
        for tau in mu.subpartitions():
            tau_count = tau.to_row_count_dict()
            binom_prod = Integer(1)
            for s in mu_count.keys():
                binom_prod *= self.binomial(mu_count[s], tau_count.get(s,0))
            remaining_sum = n - lam.element_at(0) - tau.sum
            for rho in generate_partitions(remaining_sum):
                term = self.z_inverse(rho) * self.green_polynomial_X(lam.delete_first_row(), tau.concat(rho))
                term = term.scalar_multiply(binom_prod)
                if len(rho)%2==0:
                    result = result + term
                else:
                    result = result - term
        self._green_polys[key] = result
        return result

    def green_polynomial_Q(self, lam, mu):
        return self.green_polynomial_X(lam, mu).involution()

def gp(lambda_parts):
    lam = Partition(lambda_parts)
    n = lam.sum
    mu = Partition([1]*n)
    poly = Calculator().green_polynomial_Q(lam, mu)
    return poly.poly(q)

def smaller_partitions(partition):
    result = set()
    for i in range(len(partition)):
        if partition[i] > 0:
            new_partition = partition[:]
            new_partition[i] -= 1
            if new_partition[i] == 0:
                new_partition.pop(i)
            result.add(tuple(sorted(new_partition, reverse=True)))
    return [list(p) for p in result]

def which_removed(partition, smaller_partition):
    for i, p in enumerate(partition):
        if i >= len(smaller_partition) or p - smaller_partition[i] == 1:
            return i
    raise ValueError("No single decrement/removal found")

def multiplicity(partition, i):
    part_size = partition[i]
    return partition.count(part_size)

def flag_amount(partition, p):
    dec = which_removed(partition, p)
    mult = multiplicity(partition, dec)
    return (q ** mult - 1) * (q ** (dec + 1 - mult))

def smaller_dict(partition):
    result = {}
    total = 0
    small = smaller_partitions(partition)
    for p in small:
        weight = flag_amount(partition, p) * gp(p)
        total += weight
    for p in small:
        result[tuple(p)] = flag_amount(partition, p) * gp(p) / total
    return result

# --- Sampling functions ---

def random_stabilizer_element(w):
    if hasattr(w, "list"):
        p = list(w.list())
    else:
        p = list(w)
    n = len(p)
    if min(p) == 0:
        p = [x+1 for x in p]

    inv = [0]*n
    for i, image in enumerate(p):
        inv[image-1] = i+1

    M = identity_matrix(F, n)
    for r in range(n):
        for c in range(r+1, n):
            if inv[r] <= inv[c]:
                M[r, c] = F.random_element()
            else:
                M[r, c] = F(0)
    return M

def random_nonzero_vector(k):
    N = q**k - 1
    idx = randint(1, N)
    coords = []
    for _ in range(k):
        coords.append(F(idx % q))
        idx //= q
    return vector(F, coords)

def quotient_representation(M, v):
    n = M.nrows()
    V = VectorSpace(F, n)
    v = V(v)
    
    B = [v]
    for u in V.basis():
        if len(B) == n:
            break
        if u not in span(B):
            B.append(u)
    
    P = matrix(F, B).transpose()
    Pinv = P.inverse()
    Mprime = Pinv * M * P
    Mquot = Mprime[1:, 1:]
    
    return Mquot, B

def lift(new_v, basis, original_v):
    new_v = vector(F, new_v)
    lifted = sum(coeff * b for coeff, b in zip(new_v, basis[1:]))
    return lifted

import random as py_random

def first_vector_sample(A):
    J, P = A.jordan_form(transformation=True)

    partition = []
    i = 0
    while i < J.nrows():
        size = 1
        while i+size < J.nrows() and J[i+size-1,i+size] == 1:
            size += 1
        partition.append(size)
        i += size

    sd = smaller_dict(partition)

    keys = list(sd.keys())
    weights = [float(sd[k]) for k in keys]
    sp = py_random.choices(keys, weights=weights, k=1)[0]

    dec_size = None 
    for p, s in zip(partition, sp + (0,)*(len(partition)-len(sp))):
        if p != s: 
            dec_size = p 
            break

    n = sum(partition)
    
    starts = []
    offset = 0
    for block_size in partition:
        starts.append(offset)
        offset += block_size
    
    eq_indices = [i for i, bsize in zip(starts, partition) if bsize == dec_size]
    gt_indices = [i for i, bsize in zip(starts, partition) if bsize > dec_size]
    
    v = vector(F, [F(0)]*n)
    
    if eq_indices:
        subv_eq = random_nonzero_vector(len(eq_indices))
        for pos, val in zip(eq_indices, subv_eq):
            v[pos] = val
    
    if gt_indices:
        subv_gt = random_vector(F, len(gt_indices))
        for pos, val in zip(gt_indices, subv_gt):
            v[pos] = val

    return P*v

def springer_sample_flag(A):
    n = A.nrows()
    v = first_vector_sample(A)
    flag = [v]
    if n == 1:
        return flag
    else:
        Aquot, B = quotient_representation(A, v)
        for z in springer_sample_flag(Aquot):
            flag.append(lift(z, B, v))
        return flag

def flag_to_permutation(flag):
    if not flag:
        return []
    n = len(flag[0])
    A = [list(row) for row in flag]

    pivots = []
    for i in range(len(A)):
        for r, pc in enumerate(pivots):
            if A[i][pc] != 0:
                factor = A[i][pc] / A[r][pc]
                for c in range(n):
                    A[i][c] -= factor * A[r][c]

        pc = None
        for c in range(n-1, -1, -1):
            if A[i][c] != 0:
                pc = c
                break
        if pc is None:
            raise ValueError("Dependent row encountered.")
        pivots.append(pc)

    return tuple(c+1 for c in pivots)

def next_step(w):
    u = random_stabilizer_element(w)
    x = springer_sample_flag(u)
    return flag_to_permutation(x)

# =============================================================================
# Method 1: Full Matrix Estimation + Eigenvalue Computation
# =============================================================================

def estimate_full_matrix(n, num_samples_per_state=10000):
    """
    Estimate the full transition matrix by sampling.
    Then compute eigenvalues exactly from the estimated matrix.
    """
    perms = list(itertools.permutations(range(1, n+1)))
    perm_to_idx = {p: i for i, p in enumerate(perms)}
    N = len(perms)
    
    print(f"Estimating {N}x{N} transition matrix...")
    print(f"Running {num_samples_per_state} samples from each of {N} states")
    
    counts = np.zeros((N, N))
    
    for w_idx, w in enumerate(perms):
        if True:#(w_idx + 1) % 2 == 0 or w_idx == 0:
            print(f"  State {w_idx + 1}/{N}: {w}")
        for _ in range(num_samples_per_state):
            z = next_step(w)
            z_idx = perm_to_idx[z]
            counts[w_idx, z_idx] += 1
    
    # Normalize rows
    P_est = counts / num_samples_per_state
    
    # Compute eigenvalues
    eigenvalues = np.linalg.eigvals(P_est)
    eigenvalues = sorted(eigenvalues, key=lambda x: -abs(x))
    
    print(f"\nEstimated eigenvalues (sorted by magnitude):")
    for i, ev in enumerate(eigenvalues[:min(10, len(eigenvalues))]):
        print(f"  λ_{i} = {ev:.6f}")
    
    # Second eigenvalue
    beta2 = eigenvalues[1]
    print(f"\nSecond eigenvalue β₂ ≈ {beta2:.6f}")
    
    # Get eigenvector
    eigenvalues_full, eigenvectors = np.linalg.eig(P_est)
    idx = np.argsort(-np.abs(eigenvalues_full))
    second_eigenvec = eigenvectors[:, idx[1]]
    
    # Normalize
    second_eigenvec = second_eigenvec / np.max(np.abs(second_eigenvec))
    
    print(f"\nSecond eigenvector (normalized):")
    for i, (p, v) in enumerate(zip(perms, second_eigenvec)):
        print(f"  {p}: {v:.4f}")
    
    return P_est, eigenvalues, second_eigenvec, perms

# =============================================================================
# Main
# =============================================================================

if __name__ == "__main__":
    print("="*60)
    print(f"Second Eigenvalue Estimation for GL_{n_val}(F_{q_val}) Burnside Process")
    print("="*60)
    
    # Method 1: Full matrix (most accurate for small n)
    print("\n" + "="*60)
    print("METHOD 1: Full Matrix Estimation")
    print("="*60)
    P_est, eigenvalues, eigenvec, perms = estimate_full_matrix(n_val, num_samples_per_state=30000)
    
    # For n=3, compare with known formula
    if n_val == 3:
        expected_beta2 = (2*q_val - 2) / (2*q_val + 1)
        print(f"\nComparison with known formula:")
        print(f"  Estimated β₂ = {np.real(eigenvalues[1]):.6f}")
        print(f"  Formula (2q-2)/(2q+1) = {expected_beta2:.6f}")
        print(f"  Expected eigenvector: (0, 1, -1, 1, -1, 0)")

Second Eigenvalue Estimation for GL_4(F_5) Burnside Process

METHOD 1: Full Matrix Estimation
Estimating 24x24 transition matrix...
Running 30000 samples from each of 24 states
  State 1/24: (1, 2, 3, 4)
  State 2/24: (1, 2, 4, 3)
  State 3/24: (1, 3, 2, 4)
  State 4/24: (1, 3, 4, 2)
  State 5/24: (1, 4, 2, 3)
  State 6/24: (1, 4, 3, 2)
  State 7/24: (2, 1, 3, 4)
  State 8/24: (2, 1, 4, 3)
  State 9/24: (2, 3, 1, 4)
  State 10/24: (2, 3, 4, 1)
  State 11/24: (2, 4, 1, 3)
  State 12/24: (2, 4, 3, 1)
  State 13/24: (3, 1, 2, 4)
  State 14/24: (3, 1, 4, 2)
  State 15/24: (3, 2, 1, 4)
  State 16/24: (3, 2, 4, 1)
  State 17/24: (3, 4, 1, 2)
  State 18/24: (3, 4, 2, 1)
  State 19/24: (4, 1, 2, 3)
  State 20/24: (4, 1, 3, 2)
  State 21/24: (4, 2, 1, 3)
  State 22/24: (4, 2, 3, 1)
  State 23/24: (4, 3, 1, 2)
  State 24/24: (4, 3, 2, 1)

Estimated eigenvalues (sorted by magnitude):
  λ_0 = 1.000000+0.000000j
  λ_1 = 0.764584+0.000000j
  λ_2 = 0.732929+0.000000j
  λ_3 = 0.694423+0.000000j
  λ_4 

In [43]:
# Dynkin automorphism: w ↦ w_0 w w_0
n = n_val
perms = list(itertools.permutations(range(1, n+1)))
perm_to_idx = {p: i for i, p in enumerate(perms)}

# Longest element w_0 = (n, n-1, ..., 2, 1)
w_0 = tuple(range(n, 0, -1))

def compose_perms(p1, p2):
    """Compose permutations: (p1 ∘ p2)(i) = p1[p2[i]-1]"""
    return tuple(p1[p2[i]-1] for i in range(len(p2)))

def dynkin_auto(w):
    """Apply Dynkin automorphism: w ↦ w_0 w w_0"""
    return compose_perms(compose_perms(w_0, w), w_0)

J = matrix(ZZ, len(perms), len(perms))
for i, w in enumerate(perms):
    w_auto = dynkin_auto(w)
    j = perm_to_idx[w_auto]
    J[i, j] = 1

#print(matrix(P_est))
print()
S = matrix(P_est - P_est @ J)
rounded_S = S.apply_map(lambda x: round(x, 5))
print(rounded_S)


[     0.0  0.00077      0.0  0.00333 -0.00163  0.00043 -0.00077      0.0  0.00163  0.00107      0.0  -0.0004 -0.00333      0.0 -0.00043   0.0008      0.0  -0.0007 -0.00107  -0.0008   0.0004      0.0   0.0007      0.0]
[     0.0  0.24197      0.0 -0.03797  0.20073  0.00777 -0.24197      0.0 -0.20073 -0.19897      0.0   0.0019  0.03797      0.0 -0.00777  -0.0062      0.0 -0.00773  0.19897   0.0062  -0.0019      0.0  0.00773      0.0]
[     0.0  -0.0006      0.0  0.00077 -0.00013   -7e-05   0.0006      0.0  0.00013  0.00033      0.0  0.00063 -0.00077      0.0    7e-05 -0.00033      0.0 -0.00057 -0.00033  0.00033 -0.00063      0.0  0.00057      0.0]
[     0.0 -0.03827      0.0   0.0455  0.00767  0.03937  0.03827      0.0 -0.00767  -0.0082      0.0 -0.00723  -0.0455      0.0 -0.03937 -0.03957      0.0 -0.03117   0.0082  0.03957  0.00723      0.0  0.03117      0.0]
[     0.0  0.19927      0.0  0.00787  0.20907  0.04907 -0.19927      0.0 -0.20907 -0.20987      0.0 -0.00703 -0.00787      0.0 

In [60]:
#print(matrix(P_est))
print()
S = matrix(P_est - P_est @ J)
rounded_S = S.apply_map(lambda x: round(float(x), 1))
print(rounded_S)


[ 0.0  0.0  0.0  0.0 -0.0  0.0 -0.0  0.0  0.0  0.0  0.0 -0.0 -0.0  0.0 -0.0  0.0  0.0 -0.0 -0.0 -0.0  0.0  0.0  0.0  0.0]
[ 0.0  0.2  0.0 -0.0  0.2  0.0 -0.2  0.0 -0.2 -0.2  0.0  0.0  0.0  0.0 -0.0 -0.0  0.0 -0.0  0.2  0.0 -0.0  0.0  0.0  0.0]
[ 0.0 -0.0  0.0  0.0 -0.0 -0.0  0.0  0.0  0.0  0.0  0.0  0.0 -0.0  0.0  0.0 -0.0  0.0 -0.0 -0.0  0.0 -0.0  0.0  0.0  0.0]
[ 0.0 -0.0  0.0  0.0  0.0  0.0  0.0  0.0 -0.0 -0.0  0.0 -0.0 -0.0  0.0 -0.0 -0.0  0.0 -0.0  0.0  0.0  0.0  0.0  0.0  0.0]
[ 0.0  0.2  0.0  0.0  0.2  0.0 -0.2  0.0 -0.2 -0.2  0.0 -0.0 -0.0  0.0 -0.0 -0.0  0.0 -0.0  0.2  0.0  0.0  0.0  0.0  0.0]
[ 0.0  0.0  0.0  0.0  0.0  0.2 -0.0  0.0 -0.0 -0.0  0.0 -0.0 -0.0  0.0 -0.2 -0.2  0.0 -0.2  0.0  0.2  0.0  0.0  0.2  0.0]
[ 0.0 -0.2  0.0  0.0 -0.2 -0.0  0.2  0.0  0.2  0.2  0.0 -0.0 -0.0  0.0  0.0  0.0  0.0  0.0 -0.2 -0.0  0.0  0.0 -0.0  0.0]
[ 0.0 -0.0  0.0 -0.0 -0.0  0.0  0.0  0.0  0.0 -0.0  0.0  0.0  0.0  0.0 -0.0  0.0  0.0 -0.0  0.0 -0.0 -0.0  0.0  0.0  0.0]
[ 0.0 -0.2  0.0 -0.0 -0

In [49]:
import pickle
with open('pest30k.pkl', 'wb') as f:
    pickle.dump(P_est, f)
with open('symmetrized30k.pkl', 'wb') as f:
    pickle.dump(S, f)

In [51]:
import pickle

# Read from the file
with open('symmetrized30k.pkl', 'rb') as f:
    M_loaded = pickle.load(f)

print(M_loaded)

[                    0.0   0.0007666666666666655                     0.0    0.003333333333333334   -0.001633333333333327  0.00043333333333333353  -0.0007666666666666655                     0.0    0.001633333333333327   0.0010666666666666672                     0.0 -0.00039999999999999996   -0.003333333333333334                     0.0 -0.00043333333333333353   0.0007999999999999999                     0.0  -0.0006999999999999999  -0.0010666666666666672  -0.0007999999999999999  0.00039999999999999996                     0.0   0.0006999999999999999                     0.0]
[                    0.0      0.2419666666666667                     0.0    -0.03796666666666667     0.20073333333333335    0.007766666666666667     -0.2419666666666667                     0.0    -0.20073333333333335    -0.19896666666666665                     0.0   0.0019000000000000006     0.03796666666666667                     0.0   -0.007766666666666667   -0.006199999999999999                     0.0   -0.00773333